# Workflow

## Dataset Setup for Modeling

* Data Overview
* Feature and Target Definition
* Sample Construction
* Temporal Data Split
* Data Preprocessing

## Model Development and Tuning

* Pooled Elastic Net Tuning
* Clustering: K-means
* Clustered Elastic Net Tuning

## Final Model Estimation

* Historical-average benchmark
* OLS Baseline
* Pooled Elastic Net
* Clustered Elastic Net

## Model Evaluation and Comparison

* Validation-Period Comparison
* 6-month Updating
* Test Evaluation
* Cluster Interpretation and Diagnostics

## Portfolio Construction

* Portfolio Rule
* Portfolio Performance
* Practical Trading Discussion

# 1. Dataset Setup for Modeling

This section sets up the stock-month panel that will be used throughout the project. It includes the initial dataset inspection, definition of the target and feature sets, basic sample cleaning, data-type alignment, and construction of the temporal split objects used later for model development and forecasting.

The prediction target is the forward 1-month excess return, `ret_exc_lead1m`, which is also the required target in the project brief. The JKP documentation states that `eom` represents the information available by the end of the month, and that `ret_exc_lead1m` can be predicted directly using month-\(t\) characteristics without manually lagging them again. :contentReference[oaicite:0]{index=0} :contentReference[oaicite:1]{index=1}

At this stage, we do not yet estimate any models. Instead, we define the data objects that will later feed into preprocessing, inner-loop tuning, validation forecasting, and final test evaluation.

## 1. Data Overview

### Import Module

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve,
    precision_score, recall_score, f1_score, r2_score
)
from sklearn.inspection import permutation_importance

%matplotlib inline
plt.rcParams['figure.dpi'] = 110

### Load Dataset

In [2]:
df = pd.read_parquet('jkp_data_us.parquet')

print(df.head(10))

        id        eom excntry  ret_exc_lead1m         me     be_me     at_me  \
0  10001.0 2008-07-31     USA       -0.021006  44.098861  0.712921  1.299920   
1  10001.0 2008-10-31     USA       -0.130349  36.096701  0.856006  1.617239   
2  10001.0 2008-11-30     USA        0.155960  31.225819  0.989534  1.869511   
3  10001.0 2008-12-31     USA        0.034117  35.493221  0.870561  1.644737   
4  10001.0 2009-01-31     USA        0.056075  36.533093       NaN       NaN   
5  10001.0 2009-02-28     USA       -0.080696  38.415178       NaN       NaN   
6  10001.0 2009-03-31     USA        0.044496  35.174001       NaN       NaN   
7  10001.0 2009-04-30     USA        0.002924  36.550000  0.829603  2.074391   
8  10001.0 2009-05-31     USA        0.019371  36.921918  0.821247  2.053496   
9  10001.0 2009-06-30     USA       -0.047201  36.988171  0.819776  2.049817   

    sale_me     ni_me    ocf_me  ...   roe_ch5   roa_ch5  cfoa_ch5  gmar_ch5  \
0  1.346361  0.051180 -0.028822  ...  0

In [3]:
print(df.sort_values(by='eom',ascending=False).head(5))

             id        eom excntry  ret_exc_lead1m             me     be_me  \
78970   13706.0 2024-12-31     USA             NaN     745.254671  0.780830   
129753  15056.0 2024-12-31     USA             NaN    4053.899760  0.046441   
609444  84373.0 2024-12-31     USA             NaN   16838.238968  0.392975   
852491  91233.0 2024-12-31     USA             NaN  479583.112431  0.016177   
426068  72996.0 2024-12-31     USA             NaN   10857.606427  0.434093   

           at_me   sale_me     ni_me    ocf_me  ...   roe_ch5   roa_ch5  \
78970   0.866234  0.291506 -0.068328 -0.132321  ...  0.188890  0.102250   
129753  0.092849  0.081463  0.008582  0.018010  ...  1.452276  0.507386   
609444  0.824374  0.555046  0.049886  0.074117  ... -0.003115  0.000140   
852491  0.088270  0.055027  0.025560  0.025382  ...  0.265526  0.018323   
426068  3.482285  0.510115  0.051532  0.009501  ... -0.017933 -0.002710   

        cfoa_ch5    gmar_ch5   cash_me  netis_mev  divspc1m_me  divspc12m_

### Check Shape of Dataset

Before selecting features or splitting the sample, we inspect the overall structure of the dataset:
- available columns,
- total number of observations,
- total number of variables,
- and the number of unique stocks.

This gives a quick overview of the sample size and confirms that the dataset is organized as a stock-month panel suitable for cross-sectional return prediction.

In [4]:
dfColumns = list(df.columns)
print(dfColumns)

['id', 'eom', 'excntry', 'ret_exc_lead1m', 'me', 'be_me', 'at_me', 'sale_me', 'ni_me', 'ocf_me', 'fcf_me', 'ebitda_mev', 'bev_mev', 'eq_dur', 'ival_me', 'netdebt_me', 'rd_me', 'debt_me', 'div12m_me', 'eqpo_me', 'eqnpo_me', 'gp_at', 'ope_be', 'ni_be', 'cop_at', 'op_at', 'ocf_at', 'ebit_sale', 'gp_atl1', 'ope_bel1', 'cop_atl1', 'niq_be', 'niq_at', 'pi_nix', 'op_atl1', 'ocf_at_chg1', 'niq_be_chg1', 'ret_1_0', 'ret_2_0', 'ret_3_0', 'ret_3_1', 'ret_6_0', 'ret_6_1', 'ret_9_0', 'ret_9_1', 'ret_12_0', 'ret_12_1', 'ret_12_7', 'ret_60_12', 'ret_18_1', 'ret_24_1', 'ret_24_12', 'ret_36_1', 'ret_36_12', 'ret_48_1', 'ret_60_1', 'ret_60_36', 'resff3_12_1', 'resff3_6_1', 'seas_1_1an', 'seas_1_1na', 'seas_2_5an', 'seas_2_5na', 'seas_6_10an', 'seas_6_10na', 'seas_11_15an', 'seas_11_15na', 'seas_16_20an', 'seas_16_20na', 'at_gr1', 'sale_gr1', 'capx_gr1', 'inv_gr1', 'noa_gr1a', 'ppeinv_gr1a', 'lnoa_gr1a', 'debt_gr3', 'sale_gr3', 'capx_gr3', 'emp_gr1', 'sale_emp_gr1', 'at_be', 'capx_gr2', 'inv_gr1a', 'lti_

In [5]:
print("Shape of Dataset:", df.shape)
print("")
print("Observations No.:", df.shape[0])
print("")
print("Features No.:", df.shape[1])

Shape of Dataset: (961836, 201)

Observations No.: 961836

Features No.: 201


In [6]:
id = df["id"].unique()
print("No. of Stock:", len(id))

No. of Stock: 13213


## 2. Feature and Target Definition

In [7]:
# Candidate prediction variables for first-stage screening and pooled models

prediction_features = [
    'ret_2_0', 'ret_12_0', 'ret_24_12', 'resff3_12_1', #Momentum
    'aliq_at', 'ami_126d', 'turnover_126d', 'bidaskhl_21d', #Liquidity/Frictions
    'ivol_capm_21d', 'ivol_ff3_21d', 'rvol_21d', 'beta_252d', #Risk/Volatilitye
    'gp_at', 'ocf_at', 'qmj', 'f_score', #Profitability/Quality
    'be_me', 'at_gr1', 'sale_gr1', 'capex_abn', 'me', #Value/Investment
    'netdebt_me', 'o_score', 'z_score', 'eqnetis_at' #Distress/Issuance
]

target = 'ret_exc_lead1m'

In [8]:
# Cluster variables are used only for k-means firm-type assignment

cluster_features = [
    'age', #Lifecycle
    'gp_at', 'ocf_at', 'sale_gr1', 'inv_gr1', 'dsale_dinv', #Profitability–Investment structure
    'netdebt_me', 'kz_index', 'z_score', #Financial fragility
    'niq_su', 'ni_inc8q' #Information environment
]

In [9]:
# Union of prediction and clustering variables; duplicates removed while preserving order

features = list(dict.fromkeys(prediction_features + cluster_features))

In [10]:
# Positive, highly skewed variables chosen for log1p transform

log_cols = ["me", "age"]

for c in log_cols:
    print("No. of Zero of ",c,":",(df[c]==0).sum())

No. of Zero of  me : 0
No. of Zero of  age : 12


## 3. Sample Construction

### Drop Empty Target

Because the project is a supervised return-prediction exercise, observations with missing target values cannot be used in model estimation or evaluation. We therefore remove rows with missing values in:
- stock identifier,
- month-end date,
- or the target variable `ret_exc_lead1m`.

After dropping these rows, we inspect the remaining sample and check the time coverage of the usable stock-month panel.

In [11]:
df[target].isnull().mean()*100
df_clear = df.dropna(subset=['id', 'eom', target])
df_clear[['id', 'eom', target]].isnull().mean()*100

id                0.0
eom               0.0
ret_exc_lead1m    0.0
dtype: float64

In [12]:
(df_clear[features].isnull().mean()*100).sort_values()

age               0.000000
me                0.112257
bidaskhl_21d      0.521187
ret_2_0           0.695868
ivol_capm_21d     1.236734
ivol_ff3_21d      1.236734
rvol_21d          1.236734
turnover_126d     1.920116
ami_126d          2.364067
netdebt_me        2.726231
ocf_at            3.197055
gp_at             3.232605
eqnetis_at        3.711999
beta_252d         4.083263
at_gr1            4.102942
ret_12_0          6.405114
be_me             6.788122
sale_gr1          8.460235
niq_su           11.913969
ret_24_12        13.026596
kz_index         13.517946
resff3_12_1      13.875137
aliq_at          14.640624
ni_inc8q         15.731772
z_score          16.689185
o_score          16.759650
capex_abn        20.998655
qmj              21.411288
f_score          23.108371
inv_gr1          32.568764
dsale_dinv       34.276110
dtype: float64

In [13]:
print(f"Data Time Range: {df_clear['eom'].min().date()} to {df_clear['eom'].max().date()}")

Data Time Range: 2005-01-31 to 2024-11-30


### Change Data Type and Drop Unselected Features

In [14]:
df_clear['eom'] = pd.to_datetime(df_clear['eom'])
df_clear = df_clear.sort_values(['id', 'eom']).reset_index(drop=True)

In [15]:
print(df_clear['eom'][0])

2008-07-31 00:00:00


In [16]:
df_clear = df_clear[['id', 'eom', target] + features].copy()
print(df_clear.head(5))

# missing_features = []
# for c in prediction_features+cluster_features:
#     if c not in df_clear.columns:
#         missing_features.append(c)
# print(missing_features)

        id        eom  ret_exc_lead1m   ret_2_0  ret_12_0  ret_24_12  \
0  10001.0 2008-07-31       -0.021006 -0.062580  0.133622   0.408844   
1  10001.0 2008-10-31       -0.130349 -0.153611 -0.014414   0.256010   
2  10001.0 2008-11-30        0.155960 -0.146495 -0.205725   0.289406   
3  10001.0 2008-12-31        0.034117  0.005571 -0.078204   0.334092   
4  10001.0 2009-01-31        0.056075  0.195455 -0.040900   0.295813   

   resff3_12_1   aliq_at     ami_126d turnover_126d  ... netdebt_me   o_score  \
0    -0.013293  0.582519  26.74428303    0.00185593  ...   0.236605 -2.484459   
1    -0.065312  0.667072   1.80525519    0.00196731  ...   0.327620 -2.484459   
2    -0.158375  0.667072   2.00651494    0.00117302  ...   0.378725 -2.484459   
3    -0.230158  0.667072   1.95548890    0.00124243  ...   0.333190 -2.484459   
4    -0.227538       NaN   2.01399288    0.00117742  ...        NaN       NaN   

    z_score eqnetis_at    age   inv_gr1  dsale_dinv  kz_index  niq_su  \
0  3.22

In [17]:
print(df_clear.dtypes.sort_values())
object_f = []
for f in features:
    if df_clear[f].dtypes == 'object':
        object_f.append(f)
print(object_f)
print(df_clear[object_f].head(5))
for f in features:
    df_clear[f] = pd.to_numeric(df_clear[f], errors='coerce')
print(df_clear.dtypes.sort_values())
print(df_clear[object_f].head(5))


id                       float64
kz_index                 float64
dsale_dinv               float64
inv_gr1                  float64
age                      float64
eqnetis_at               float64
z_score                  float64
o_score                  float64
netdebt_me               float64
me                       float64
capex_abn                float64
sale_gr1                 float64
at_gr1                   float64
be_me                    float64
f_score                  float64
qmj                      float64
ocf_at                   float64
gp_at                    float64
eom               datetime64[ms]
ret_exc_lead1m           float64
ret_2_0                  float64
ret_12_0                 float64
ret_24_12                float64
resff3_12_1              float64
niq_su                   float64
aliq_at                  float64
ni_inc8q                 float64
turnover_126d             object
bidaskhl_21d              object
ivol_capm_21d             object
ivol_ff3_2

## 4. Temporal Data Split

This subsection creates the temporal data objects used in the rest of the notebook.

First, we define the **inner split** used for model development:
- inner train: 2005-01 to 2012-12
- inner validation: 2013-01 to 2015-12

This inner split is used to tune model hyperparameters such as:
- pooled Elastic Net regularization settings,
- clustered Elastic Net regularization settings,
- and the number of clusters \(K\).

Second, we predefine the **6-month expanding-window update blocks** for:
- the official validation period,
- and the official test period.

Each update block contains:
- an expanding estimation sample using only data available up to the refit date,
- and the next 6-month forecast block to be predicted.

This setup allows the later forecasting sections to follow a strict time-respecting protocol with no look-ahead bias, while keeping the implementation organized and reproducible.

In [18]:
# def split_train_valid_test(df: pd.DataFrame, validation_cutoff, test_cutoff):
#     """
#     Official project split:
#     Train: 2005-01 to 2015-12
#     Valid: 2016-01 to 2018-12
#     Test:  2019-01 to 2024-12
#     """
#     train_mask = (df['eom'] < validation_cutoff)
#     valid_mask = (df['eom'] >= validation_cutoff) & (df['eom'] < test_cutoff)
#     test_mask  = (df['eom'] >= test_cutoff)

#     train_df = df.loc[train_mask].copy()
#     valid_df = df.loc[valid_mask].copy()
#     test_df  = df.loc[test_mask].copy()

#     return train_df, valid_df, test_df

def split_df_2(df: pd.DataFrame, cutoff, end):

    # Dictionary: refit_date -> (estimation_window, next_6_month_prediction_block)

    train_mask = (df['eom'] < cutoff)
    valid_mask = (df['eom'] >= cutoff) & (df['eom'] < end)

    train_df = df.loc[train_mask].copy()
    valid_df = df.loc[valid_mask].copy()

    return train_df, valid_df

In [41]:
# Inner split used only for model development:
# inner train fits candidate models; inner validation tunes K, alpha, and lambda.
# This keeps the outer validation period cleaner for finalized model comparison.

inner_train_df, inner_valid_df = split_df_2(df_clear, '2013-01-01', '2016-01-01')

In [42]:
def expanding_window_6_months(date):
    if date[6] == '1':
        new_date = str(int(date[0:4]))+'-'+'07'+date[7:]
    if date[6] == '7':
        new_date = str(int(date[0:4])+1)+'-'+'01'+date[7:]
    return new_date

def date_updater(date=pd.to_datetime('2019-01-01'), frequency=1, unit='months'):

    """
    unit: days, weeks, months, years
    
    e.g. 1 months, 6 months
    """

    date = pd.to_datetime(date)

    if unit == 'days':
        date += pd.Timedelta(days=frequency)

    if unit == 'weeks':
        date += pd.Timedelta(weeks=frequency)
    
    if unit == 'months':
        date += pd.DateOffset(months=frequency)
    
    if unit == 'years':
        date += pd.DateOffset(years=frequency)

    return date

# Build 6-month expanding-window refit blocks:
# each entry contains (estimation window up to refit date (start_date), next 6-month prediction block).

def expanding_window(df, start_date, end_date, frequency=6, unit='months'):

    update_set = {} # Today: (train_df, pred_df)

    while start_date < end_date:
        next_date = date_updater(start_date, frequency, unit)
        update_set[start_date] = split_df_2(df, start_date, next_date)
        start_date = next_date
    
    return update_set

In [43]:
# Build 6-month expanding-window refit blocks:
# each entry contains (estimation window up to refit date, next 6-month prediction block).

valid_update_set = expanding_window(df_clear, pd.to_datetime('2016-01-01'), pd.to_datetime('2019-01-01'), 6, 'months')
test_update_set = expanding_window(df_clear, pd.to_datetime('2019-01-01'), pd.to_datetime('2025-01-01'), 6, 'months')

In [44]:
def summarize_split(name, df):
    print(f"{name}: {df['eom'].min().date()} to {df['eom'].max().date()}, rows={len(df):,}, ids={df['id'].nunique():,}")

summarize_split("Inner train", inner_train_df)
summarize_split("Inner valid", inner_valid_df)
print("")

for k, (train, valid) in valid_update_set.items():
    summarize_split(f"Train {k}", train)
    summarize_split(f"Valid {k}", valid)
    print("")

for k, (train, test) in test_update_set.items():
    summarize_split(f"Train {k}", train)
    summarize_split(f"Test  {k}", test)
    print("")

Inner train: 2005-01-31 to 2012-12-31, rows=383,570, ids=6,869
Inner valid: 2013-01-31 to 2015-12-31, rows=133,564, ids=4,846

Train 2016-01-01 00:00:00: 2005-01-31 to 2015-12-31, rows=517,134, ids=7,842
Valid 2016-01-01 00:00:00: 2016-01-31 to 2016-06-30, rows=22,602, ids=3,962

Train 2016-07-01 00:00:00: 2005-01-31 to 2016-06-30, rows=539,736, ids=7,912
Valid 2016-07-01 00:00:00: 2016-07-31 to 2016-12-31, rows=22,055, ids=3,903

Train 2017-01-01 00:00:00: 2005-01-31 to 2016-12-31, rows=561,791, ids=8,038
Valid 2017-01-01 00:00:00: 2017-01-31 to 2017-06-30, rows=21,882, ids=3,846

Train 2017-07-01 00:00:00: 2005-01-31 to 2017-06-30, rows=583,673, ids=8,167
Valid 2017-07-01 00:00:00: 2017-07-31 to 2017-12-31, rows=22,017, ids=3,889

Train 2018-01-01 00:00:00: 2005-01-31 to 2017-12-31, rows=605,690, ids=8,310
Valid 2018-01-01 00:00:00: 2018-01-31 to 2018-06-30, rows=21,836, ids=3,855

Train 2018-07-01 00:00:00: 2005-01-31 to 2018-06-30, rows=627,526, ids=8,469
Valid 2018-07-01 00:00:00:

## 5. Data Preprocessing

In [45]:
# Custom preprocessing pipeline:
# fit all preprocessing objects on the training segment only, then apply them to later blocks.
# This avoids look-ahead bias in winsorization, imputation, and standardization.

class StockPreprocessor:
    """
    Preprocessing pipeline for stock characteristics.

    Steps:
    1. impute missing values with training medians
    2. log(1+x) transform selected variables
    3. winsorize using training-set quantiles only
    4. standardize using training mean/std
    """

    def __init__(
        self,
        feature_cols,
        target_col,
        log1p_cols=None,
        id_col = 'id',
        date_col = 'eom',
        winsor_lower=0.01,
        winsor_upper=0.99
    ):
        self.id_col = id_col
        self.date_col = date_col
        self.feature_cols = feature_cols
        self.target_col = target_col
        self.log1p_cols = log1p_cols if log1p_cols is not None else []
        self.winsor_lower = winsor_lower
        self.winsor_upper = winsor_upper

        # These objects are learned from training data only
        self.X_lower_bounds_ = None
        self.X_upper_bounds_ = None
        self.y_lower_bounds_ = None
        self.y_upper_bounds_ = None
        self.imputer_ = None
        self.scaler_ = None

    def _apply_log1p(self, df):
        
        # Apply log1p only to selected positive, highly skewed variables.
        # Their transformed names become log_<variable> in the processed output.

        out = df.copy()

        for col in self.log1p_cols:
            if col in out.columns:
                out[col] = np.log1p(out[col])
        
        return out

    def _winsorize_with_fitted_bounds(self, df):
        
        # Winsorization bounds are estimated from the training segment only
        # and then applied transformation to the corresponding validation / test block.

        X_wins = np.clip(df[self.feature_cols].values.astype(float), self.X_lower_bounds_, self.X_upper_bounds_)

        y_wins = df[self.target_col] #.clip(self.y_lower_bounds_, self.y_upper_bounds_)

        return X_wins, y_wins

    def fit(self, train_df):
        """
        Fit preprocessing objects using training data only.
        """
        X = train_df[self.feature_cols].copy()
        y = train_df[self.target_col].copy()

        X = self._apply_log1p(X)

        # Step 1: compute winsorization bounds from training only
        self.X_lower_bounds_ = X.quantile(self.winsor_lower).values.astype(float)
        self.X_upper_bounds_ = X.quantile(self.winsor_upper).values.astype(float)

        self.y_lower_bounds_ = float(y.quantile(self.winsor_lower))
        self.y_upper_bounds_ = float(y.quantile(self.winsor_upper))

        df = pd.concat([y, X], axis=1)

        # Apply winsorization to the training data
        X, y = self._winsorize_with_fitted_bounds(df)

        # Step 2: fit median imputer on training only
        self.imputer_ = SimpleImputer(strategy="median")

        X_imputed = pd.DataFrame(self.imputer_.fit_transform(X), columns=self.feature_cols, index=train_df.index)

        # Step 3: fit scaler on training only
        self.scaler_ = StandardScaler()
        self.scaler_.fit(X_imputed)

        return self

    def transform(self, df):
        """
        Apply fitted preprocessing objects to any new dataset
        such as validation or test.
        """
        X = df[self.feature_cols].copy()
        y = df[self.target_col].copy()

        X = self._apply_log1p(X)
        
        out = pd.concat([y, X], axis=1)

        # Apply the same winsorization bounds learned from training
        X, y = self._winsorize_with_fitted_bounds(out)

        # Apply the same training-fitted median imputer
        X_imputed = pd.DataFrame(self.imputer_.transform(X), columns=self.feature_cols, index=df.index)

        # Apply the same training-fitted scaler
        X_scaled = self.scaler_.transform(X_imputed)

        # Put transformed features back into a DataFrame
        X_out = pd.DataFrame(X_scaled, columns=self.feature_cols, index=out.index)

        # Return original dataset with transformed feature columns replaced
        out = pd.concat([df[[self.id_col, self.date_col]].copy(), y, X_out], axis=1)

        # After preprocessing, 'me' and 'age' become 'log_me' and 'log_age'
        for x in self.log1p_cols:
            if x in out.columns:
                out = out.rename(columns={x: f"log_{x}"})
        return out

    def fit_transform(self, train_df):
        """
        Convenience method: fit on training data and transform training data.
        """
        self.fit(train_df)
        return self.transform(train_df)

### Preprocessing Data

In [46]:
# Fit preprocessing objects on inner training data only.

SP_inner = StockPreprocessor(feature_cols=features, target_col=target, log1p_cols=log_cols)
SP_inner.fit(inner_train_df)

In [47]:
# Apply the same fitted transforms to inner validation to avoid look-ahead bias.

inner_train_df = SP_inner.transform(inner_train_df)
inner_valid_df = SP_inner.transform(inner_valid_df)


In [48]:
# All downstream model code should use model_features, not raw features

model_features = features.copy()
for i in log_cols:
    model_features[model_features.index(i)] = f"log_{i}"
print(model_features)

['ret_2_0', 'ret_12_0', 'ret_24_12', 'resff3_12_1', 'aliq_at', 'ami_126d', 'turnover_126d', 'bidaskhl_21d', 'ivol_capm_21d', 'ivol_ff3_21d', 'rvol_21d', 'beta_252d', 'gp_at', 'ocf_at', 'qmj', 'f_score', 'be_me', 'at_gr1', 'sale_gr1', 'capex_abn', 'log_me', 'netdebt_me', 'o_score', 'z_score', 'eqnetis_at', 'log_age', 'inv_gr1', 'dsale_dinv', 'kz_index', 'niq_su', 'ni_inc8q']


In [49]:
log_prediction_features = prediction_features.copy()
for i in log_cols:
    if i in log_prediction_features:
        log_prediction_features[log_prediction_features.index(i)] = f"log_{i}"

log_cluster_features = cluster_features.copy()
for i in log_cols:
    if i in log_cluster_features:
        log_cluster_features[log_cluster_features.index(i)] = f"log_{i}"

print(log_prediction_features)
print(log_cluster_features)

['ret_2_0', 'ret_12_0', 'ret_24_12', 'resff3_12_1', 'aliq_at', 'ami_126d', 'turnover_126d', 'bidaskhl_21d', 'ivol_capm_21d', 'ivol_ff3_21d', 'rvol_21d', 'beta_252d', 'gp_at', 'ocf_at', 'qmj', 'f_score', 'be_me', 'at_gr1', 'sale_gr1', 'capex_abn', 'log_me', 'netdebt_me', 'o_score', 'z_score', 'eqnetis_at']
['log_age', 'gp_at', 'ocf_at', 'sale_gr1', 'inv_gr1', 'dsale_dinv', 'netdebt_me', 'kz_index', 'z_score', 'niq_su', 'ni_inc8q']


In [50]:
# At each refit date, we re-estimate preprocessing objects, centroids, and coefficients
# using only data available up to that date, then forecast the next 6 months.

# Dictionary of 6-month expanding-window refit blocks for the validation stage:
# key = refit date; value = (past estimation sample, next 6-month forecast block)

valid_update_window = {}
for start_date in valid_update_set.keys():
    SP = StockPreprocessor(feature_cols=features, target_col=target, log1p_cols=log_cols)
    SP.fit(valid_update_set[start_date][0])
    valid_update_window[start_date] = [SP.transform(valid_update_set[start_date][0]),SP.transform(valid_update_set[start_date][1])]

In [51]:
# Dictionary of 6-month expanding-window refit blocks for the test stage:
# key = refit date; value = (past estimation sample, next 6-month forecast block)

test_update_window = {}
for start_date in test_update_set.keys():
    SP = StockPreprocessor(feature_cols=features, target_col=target, log1p_cols=log_cols)
    SP.fit(test_update_set[start_date][0])
    test_update_window[start_date] = [SP.transform(test_update_set[start_date][0]),SP.transform(test_update_set[start_date][1])]

### Check Preprocessed Data

In [52]:
mean_pre_valid = 0
for x in valid_update_window.keys():
    mean_pre_valid += valid_update_window[x][0][model_features].mean()

print("Train-Valid Set Mean:\n", mean_pre_valid/len(valid_update_window), sep='')

Train-Valid Set Mean:
ret_2_0          1.286920e-17
ret_12_0         7.610723e-18
ret_24_12       -2.606771e-17
resff3_12_1     -7.744038e-18
aliq_at         -3.027717e-16
ami_126d        -9.583867e-18
turnover_126d    5.625222e-17
bidaskhl_21d    -9.168453e-17
ivol_capm_21d    8.436915e-18
ivol_ff3_21d     1.159910e-16
rvol_21d        -5.511130e-17
beta_252d       -2.249587e-16
gp_at            1.372538e-16
ocf_at           5.695632e-18
qmj             -6.562501e-18
f_score         -6.738544e-18
be_me            1.003066e-16
at_gr1          -4.361215e-17
sale_gr1         3.826915e-17
capex_abn        1.630748e-17
log_me           9.064903e-19
netdebt_me      -1.028901e-17
o_score         -1.837070e-16
z_score         -1.720169e-17
eqnetis_at       3.080197e-17
log_age          1.968154e-17
inv_gr1          2.078596e-20
dsale_dinv      -5.058523e-18
kz_index         6.554987e-18
niq_su           1.420841e-17
ni_inc8q        -3.897035e-18
dtype: float64


In [53]:
std_pre_valid = 0
for x in valid_update_window.keys():
    std_pre_valid += valid_update_window[x][0][model_features].std(ddof=0)

print("Train-Valid Set Std:\n", std_pre_valid/len(valid_update_window), sep='')

Train-Valid Set Std:
ret_2_0          1.0
ret_12_0         1.0
ret_24_12        1.0
resff3_12_1      1.0
aliq_at          1.0
ami_126d         1.0
turnover_126d    1.0
bidaskhl_21d     1.0
ivol_capm_21d    1.0
ivol_ff3_21d     1.0
rvol_21d         1.0
beta_252d        1.0
gp_at            1.0
ocf_at           1.0
qmj              1.0
f_score          1.0
be_me            1.0
at_gr1           1.0
sale_gr1         1.0
capex_abn        1.0
log_me           1.0
netdebt_me       1.0
o_score          1.0
z_score          1.0
eqnetis_at       1.0
log_age          1.0
inv_gr1          1.0
dsale_dinv       1.0
kz_index         1.0
niq_su           1.0
ni_inc8q         1.0
dtype: float64


In [54]:
for x in valid_update_window.keys():
    print('Shape of train-valid set', x,':', valid_update_window[x][0].shape,'/', valid_update_window[x][1].shape)

Shape of train-valid set 2016-01-01 00:00:00 : (517134, 34) / (22602, 34)
Shape of train-valid set 2016-07-01 00:00:00 : (539736, 34) / (22055, 34)
Shape of train-valid set 2017-01-01 00:00:00 : (561791, 34) / (21882, 34)
Shape of train-valid set 2017-07-01 00:00:00 : (583673, 34) / (22017, 34)
Shape of train-valid set 2018-01-01 00:00:00 : (605690, 34) / (21836, 34)
Shape of train-valid set 2018-07-01 00:00:00 : (627526, 34) / (22303, 34)


In [55]:
mean_pre_test = 0
for x in test_update_window.keys():
    mean_pre_test += test_update_window[x][0][model_features].mean()

print("Train-Test Set Mean:\n", mean_pre_test/len(test_update_window), sep='')

Train-Test Set Mean:
ret_2_0          3.946880e-18
ret_12_0        -1.353370e-17
ret_24_12       -1.598306e-17
resff3_12_1      3.345250e-18
aliq_at         -9.751458e-17
ami_126d         1.348877e-18
turnover_126d   -4.554805e-17
bidaskhl_21d    -1.387098e-16
ivol_capm_21d    8.232111e-17
ivol_ff3_21d     6.040537e-17
rvol_21d         2.400978e-16
beta_252d       -8.843877e-17
gp_at           -8.015164e-18
ocf_at          -9.554327e-18
qmj              4.789423e-18
f_score          4.390001e-17
be_me            5.428299e-17
at_gr1          -1.567398e-17
sale_gr1         2.833934e-17
capex_abn        2.106916e-18
log_me           4.643441e-16
netdebt_me      -6.739080e-18
o_score         -2.327069e-16
z_score         -7.156262e-17
eqnetis_at       1.420403e-17
log_age          4.193039e-16
inv_gr1          7.373472e-18
dsale_dinv      -3.286332e-18
kz_index         1.926313e-18
niq_su           8.003770e-18
ni_inc8q        -4.793127e-18
dtype: float64


In [56]:
std_pre_test = 0
for x in test_update_window.keys():
    std_pre_test += test_update_window[x][0][model_features].std(ddof=0)

print("Train-Test Set Std:\n", std_pre_test/len(test_update_window), sep='')

Train-Test Set Std:
ret_2_0          1.0
ret_12_0         1.0
ret_24_12        1.0
resff3_12_1      1.0
aliq_at          1.0
ami_126d         1.0
turnover_126d    1.0
bidaskhl_21d     1.0
ivol_capm_21d    1.0
ivol_ff3_21d     1.0
rvol_21d         1.0
beta_252d        1.0
gp_at            1.0
ocf_at           1.0
qmj              1.0
f_score          1.0
be_me            1.0
at_gr1           1.0
sale_gr1         1.0
capex_abn        1.0
log_me           1.0
netdebt_me       1.0
o_score          1.0
z_score          1.0
eqnetis_at       1.0
log_age          1.0
inv_gr1          1.0
dsale_dinv       1.0
kz_index         1.0
niq_su           1.0
ni_inc8q         1.0
dtype: float64


In [57]:
for x in test_update_window.keys():
    print('Shape of train-test set', x,':', test_update_window[x][0].shape,'/', test_update_window[x][1].shape)

Shape of train-test set 2019-01-01 00:00:00 : (649829, 34) / (22387, 34)
Shape of train-test set 2019-07-01 00:00:00 : (672216, 34) / (22564, 34)
Shape of train-test set 2020-01-01 00:00:00 : (694780, 34) / (22970, 34)
Shape of train-test set 2020-07-01 00:00:00 : (717750, 34) / (23690, 34)
Shape of train-test set 2021-01-01 00:00:00 : (741440, 34) / (25377, 34)
Shape of train-test set 2021-07-01 00:00:00 : (766817, 34) / (26983, 34)
Shape of train-test set 2022-01-01 00:00:00 : (793800, 34) / (27618, 34)
Shape of train-test set 2022-07-01 00:00:00 : (821418, 34) / (27665, 34)
Shape of train-test set 2023-01-01 00:00:00 : (849083, 34) / (26537, 34)
Shape of train-test set 2023-07-01 00:00:00 : (875620, 34) / (25402, 34)
Shape of train-test set 2024-01-01 00:00:00 : (901022, 34) / (24482, 34)
Shape of train-test set 2024-07-01 00:00:00 : (925504, 34) / (19647, 34)


In [58]:
print("Inner Train Set Mean:\n",inner_train_df[model_features].mean(), sep='')
print("Inner Validation Set Mean\n",inner_valid_df[model_features].mean(), sep='')

Inner Train Set Mean:
ret_2_0          2.697161e-17
ret_12_0         4.149479e-18
ret_24_12        5.705534e-17
resff3_12_1      2.074740e-18
aliq_at          2.185886e-18
ami_126d         2.252574e-17
turnover_126d    4.742262e-18
bidaskhl_21d     2.276286e-16
ivol_capm_21d    1.810951e-16
ivol_ff3_21d    -1.049225e-16
rvol_21d        -3.095808e-16
beta_252d        1.351545e-16
gp_at           -8.773185e-17
ocf_at          -2.267394e-17
qmj              2.482278e-17
f_score         -1.653864e-16
be_me            1.858374e-16
at_gr1          -9.380787e-17
sale_gr1        -7.706176e-17
capex_abn        1.155926e-17
log_me           5.832982e-16
netdebt_me      -3.171388e-17
o_score          6.046384e-17
z_score         -1.072937e-16
eqnetis_at      -1.837626e-17
log_age          5.145354e-16
inv_gr1         -5.446191e-18
dsale_dinv       2.978733e-17
kz_index        -4.208757e-17
niq_su          -8.743545e-18
ni_inc8q         3.971644e-17
dtype: float64
Inner Validation Set Mean
ret_2_0

In [59]:
print("Inner Trainning Set Std:\n",inner_train_df[model_features].std(ddof=0), sep='')
print("Inner Validation Set Std\n",inner_valid_df[model_features].std(ddof=0), sep='')

Inner Trainning Set Std:
ret_2_0          1.0
ret_12_0         1.0
ret_24_12        1.0
resff3_12_1      1.0
aliq_at          1.0
ami_126d         1.0
turnover_126d    1.0
bidaskhl_21d     1.0
ivol_capm_21d    1.0
ivol_ff3_21d     1.0
rvol_21d         1.0
beta_252d        1.0
gp_at            1.0
ocf_at           1.0
qmj              1.0
f_score          1.0
be_me            1.0
at_gr1           1.0
sale_gr1         1.0
capex_abn        1.0
log_me           1.0
netdebt_me       1.0
o_score          1.0
z_score          1.0
eqnetis_at       1.0
log_age          1.0
inv_gr1          1.0
dsale_dinv       1.0
kz_index         1.0
niq_su           1.0
ni_inc8q         1.0
dtype: float64
Inner Validation Set Std
ret_2_0          0.846673
ret_12_0         0.884040
ret_24_12        0.809398
resff3_12_1      1.008026
aliq_at          1.152994
ami_126d         0.973833
turnover_126d    0.914973
bidaskhl_21d     0.765433
ivol_capm_21d    0.878140
ivol_ff3_21d     0.869425
rvol_21d         0.79704

In [60]:
print("Inner Trainning Set:",inner_train_df.shape)
print("Inner Validation Set:",inner_valid_df.shape)

Inner Trainning Set: (383570, 34)
Inner Validation Set: (133564, 34)


# 2. Model Development and Tuning

This section finalizes the model specification using the processed data and the inner split only. Since preprocessing and date splitting have already been completed, we begin directly with model-building choices.

Our goal is to compare a non-clustered regularized benchmark against a cluster-conditional regularized model. Because the pooled Elastic Net and clustered Elastic Net use different design matrices, they are tuned separately. The pooled Elastic Net uses only base predictors, while the clustered Elastic Net uses base predictors, cluster dummies, and predictor-by-cluster interactions.

At the end of this section, we fix:
- the tuned hyperparameters for pooled Elastic Net,
- the chosen number of clusters \(K^*\),
- the tuned hyperparameters for clustered Elastic Net.

These hyperparameters remain fixed afterward. In later forecasting stages, we only update preprocessing objects, cluster centroids, and model coefficients every 6 months.

## 1. Pooled Elastic Net Tuning

### Objective
Tune the regularized non-clustered benchmark model.

### Model idea
The pooled Elastic Net uses the selected prediction variables only, with no cluster identifiers and no interaction terms. It serves as the regularized benchmark against which we later compare the clustered Elastic Net.

### Inputs
The pooled predictor set is the theory-motivated candidate set covering:
- momentum,
- liquidity/frictions,
- risk/volatility,
- profitability/quality,
- value/investment,
- distress/issuance.

Confirmed candidate predictors include:
- Momentum: `ret_2_0`, `ret_12_0`, `ret_24_12`, `resff3_12_1`
- Liquidity/Frictions: `aliq_at`, `ami_126d`, `turnover_126d`, `bidaskhl_21d`
- Risk/Volatility: `ivol_capm_21d`, `ivol_ff3_21d`, `rvol_21d`, `beta_252d`
- Profitability/Quality: `gp_at`, `ocf_at`, `qmj`, `f_score`
- Value/Investment: `be_me`, `at_gr1`, `sale_gr1`, `capex_abn`, `me`
- Distress/Issuance: `netdebt_me`, `o_score`, `z_score`, `eqnetis_at`

### Inner tuning task
Using the inner train / inner validation split, we tune:
- `alpha_pooled`
- `lambda_pooled`

### What this section should do
- construct the pooled Elastic Net design matrix,
- define the tuning grid for `alpha` and `lambda`,
- fit the pooled Elastic Net on the inner training sample,
- evaluate predictive performance on the inner validation sample,
- select the best pooled Elastic Net hyperparameters.

### Why we do it
This model provides a fair regularized benchmark. It allows us to distinguish:
- the effect of regularization alone,
from
- the additional effect of introducing cluster-conditioned predictor slopes.

### Useful visualizations
- validation-score heatmap across `alpha` and `lambda`,
- coefficient-path plot against `lambda`,
- table of active predictors under the selected pooled Elastic Net.

## 2. Clustering: K-means

### Objective
Group firms into economically meaningful clusters before estimating the cluster-conditional prediction model.

### Clustering variables
We use a smaller firm-type variable set designed to capture:
- lifecycle,
- profitability-investment structure,
- financial fragility,
- information environment.

Confirmed clustering variables:
- `age`
- `gp_at`, `ocf_at`, `sale_gr1`, `inv_gr1`, `dsale_dinv`
- `netdebt_me`, `kz_index`, `z_score`
- `niq_su`, `ni_inc8q`

### Method
We use k-means clustering with candidate cluster counts:
- \(K = 2, 3, 4, 5, ...\)

### What this section should do
- construct the clustering feature matrix from the processed clustering variables,
- define the candidate range of `K`,
- fit k-means on the inner training sample,
- define the procedure for assigning later observations to the nearest centroids.

### Why we do it
Our project hypothesis is that firms may differ systematically in the way characteristics map into future returns. Clustering provides a data-driven but interpretable firm-type classification that can be used to allow slope heterogeneity in the final prediction model.

### Useful visualizations for cluster construction
These figures are diagnostic tools for the clustering procedure itself:
- **Cluster-size bar chart** for each candidate \(K\), to check whether clusters are extremely unbalanced
- **Within-cluster variation / elbow plot** across candidate \(K\), as a simple diagnostic for whether adding clusters materially reduces within-cluster dispersion
- **Centroid summary table** for each candidate \(K\), to inspect whether the resulting cluster profiles are distinct before moving to the final model

## 3. Clustered Elastic Net Tuning

### Objective
Tune the main model of the project: the cluster-conditional Elastic Net.

### Model idea
The clustered Elastic Net uses:
- base prediction variables,
- cluster dummies,
- predictor-by-cluster interactions.

This allows the predictive effect of each characteristic to vary across firm clusters.

### Inner tuning task
Using the inner train / inner validation split, we tune:
- \(K\)
- `alpha_clustered`
- `lambda_clustered`

### What this section should do
For each candidate combination of `K`, `alpha`, and `lambda`:
1. fit k-means on the inner training sample,
2. assign inner-validation observations to the nearest centroids,
3. construct the final clustered design matrix,
4. fit the clustered Elastic Net on the inner training sample,
5. evaluate predictive performance on the inner validation sample.

At the end of this section, we select:
- \(K^*\),
- `alpha_clustered*`,
- `lambda_clustered*`.

### Why we do it
Once cluster dummies and predictor-by-cluster interactions are added, the number of parameters grows quickly. Elastic Net is used here to shrink the expanded interaction design and reduce the effective number of active parameters while preserving flexibility in the final forecasting model.

### Useful visualizations
- validation-score heatmap for clustered Elastic Net,
- validation-score comparison across candidate `K`,
- number of active coefficients under the chosen clustered model,
- heatmap of top interaction coefficients.

# 3. Final Model Estimation

This section estimates the finalized comparison models using the tuned specifications from the previous section. No further hyperparameter search is conducted here.

The final comparison set includes:
- Historical-average benchmark,
- Pooled OLS,
- Pooled Elastic Net,
- Clustered Elastic Net.

All models will later be run under the same 6-month expanding-window update protocol.

## 1. Historical-average benchmark

### Objective
Estimate the required benchmark model from the project brief.

### Model idea
For each stock, predict next-month excess return as the stock’s own historical average excess return, using only observations available up to the current refit date.

### What this section should do
- define the stock-level historical-average forecasting rule,
- generate validation and test predictions using the current estimation window,
- update the benchmark every 6 months in the main specification.

### Why we do it
This is the required benchmark model and gives a simple stock-level baseline before introducing cross-sectional predictors or machine-learning structure.

### Useful visualizations
- cumulative return curve for the benchmark portfolio,
- histogram of benchmark predictions,
- summary statistics of stock-level historical means.

## 2. OLS Baseline

### Objective
Estimate the pooled linear benchmark without regularization or cluster structure.

### Model idea
Use the processed predictor set in a pooled cross-sectional OLS regression.

### What this section should do
- build the pooled OLS design matrix,
- fit OLS on the current estimation window,
- generate stock-level predictions for the next forecast block.

### Why we do it
This is the required linear baseline and provides the benchmark for judging whether regularization and cluster-conditioning add value.

### Useful visualizations
- coefficient table for a selected refit date,
- coefficient stability plot across refit windows,
- predicted-vs-realized scatterplot for a validation block.

## 3. Pooled Elastic Net

### Objective
Estimate the tuned regularized non-clustered benchmark.

### Model idea
Use the same base predictors as the clustered model, but without cluster dummies or interaction terms.

### Hyperparameters
Use the tuned:
- `alpha_pooled*`
- `lambda_pooled*`

### What this section should do
- fit pooled Elastic Net on each estimation window using the fixed pooled hyperparameters,
- generate stock-level predictions for the next forecast block.

### Why we do it
This model isolates the effect of regularization. Comparing it with clustered Elastic Net tells us whether cluster-conditioning adds value beyond a non-clustered regularized benchmark.

### Useful visualizations
- active-coefficient count over time,
- comparison of active predictors vs OLS coefficients,
- coefficient-path plot for the selected pooled model.

## 4. Clustered Elastic Net

### Objective
Estimate the main model of the project using the tuned clustered specification.

### Model idea
The final clustered Elastic Net includes:
- base predictors,
- cluster dummies,
- predictor-by-cluster interactions.

### Hyperparameters
Use the tuned:
- \(K^*\)
- `alpha_clustered*`
- `lambda_clustered*`

### What this section should do
- re-estimate k-means centroids with fixed \(K^*\),
- assign cluster labels,
- construct cluster dummies and predictor-by-cluster interactions,
- fit clustered Elastic Net with the fixed clustered hyperparameters,
- generate stock-level predictions for the next forecast block.

### Why we do it
This is the core model used to test whether predictor effects differ across firm types and whether allowing such heterogeneity improves prediction and portfolio performance.

### Useful visualizations
- active interaction count over time,
- top interaction coefficients table,
- coefficient heatmap by predictor and cluster,
- comparison of predicted-return distributions from pooled vs clustered Elastic Net.

# 4. Model Evaluation and Comparison

This section compares all finalized models under the same forecasting protocol. We use the official validation period for model comparison and the official test period for final out-of-sample predictive evaluation. The same 6-month expanding-window update logic is used throughout so that all models are compared fairly.

## 1. Validation-Period Comparison

### Objective
Compare finalized models on the official validation period before moving to the untouched test period.

### Validation period
- 2016-01 to 2018-12

### Models compared
- Historical-average benchmark
- Pooled OLS
- Pooled Elastic Net
- Clustered Elastic Net

### What this section should do
- run all finalized models on the validation update blocks,
- collect stock-level predictions,
- compare predictive performance across models.

### Main comparison questions
- Does regularization improve performance relative to pooled OLS?
- Does cluster-conditioning improve performance relative to pooled Elastic Net?

### Useful visualizations
- validation \(R^2\) comparison table,
- bar chart of validation \(R^2\),
- rolling validation performance by refit block.

## 2. 6-month Updating

### Objective
Implement the agreed forecasting protocol.

### Update rule
- expanding window,
- refit every 6 months.

### What updates every 6 months
- preprocessing objects,
- cluster centroids,
- model coefficients.

### What remains fixed
- clustering algorithm,
- \(K^*\),
- pooled Elastic Net hyperparameters,
- clustered Elastic Net hyperparameters,
- predictor sets,
- model structure.

### Why we do it
This allows the models to adapt to newly available data while preserving a fixed design and avoiding repeated hyperparameter re-optimization in the main specification.

### Useful visualizations
- timeline of refit dates and forecast blocks,
- diagram of one estimation window and its next 6-month prediction block,
- summary table of all refit windows.

## 3. Test Evaluation

### Objective
Evaluate final out-of-sample predictive performance on the official test period.

### Test period
- 2019-01 to 2024-12

### What this section should do
- run the same 6-month forecasting protocol used in validation,
- generate monthly stock-level predictions for each model,
- compute test out-of-sample \(R^2\),
- compare predictive ranking ability across models.

### Required predictive output
Report test \(R^2\) for:
- Historical average
- Pooled OLS
- Pooled Elastic Net
- Clustered Elastic Net

### Useful visualizations
- test \(R^2\) comparison chart,
- predicted-return decile vs realized-return plot,
- predicted vs realized return scatter or binned plot.

## 4. Cluster Interpretation and Diagnostics

### Objective
Interpret the selected clustering structure and the economic meaning of the final clustered model.

### What this section should do
- summarize the selected centroids using the clustering variables,
- label each cluster economically,
- compare cluster sizes,
- examine how the final active interaction terms differ across clusters,
- assess whether the final clustered model tells a coherent finance story.

### Why this matters
The clustered model is valuable only if it improves prediction in a way that is also economically interpretable. This section connects the ML results back to a finance story.

### Useful visualizations for cluster interpretation
These figures are for understanding the *meaning* of the chosen clusters, not for constructing them:
- **Centroid heatmap** across the clustering variables, to show the economic profile of each selected cluster
- **Cluster-size distribution** under the selected \(K^*\), to show how the final partition is distributed across firms
- **Boxplots or violin plots of key clustering variables by cluster**, to show how the selected clusters differ along important economic dimensions
- **PCA scatter plot colored by cluster**, used only as a low-dimensional visualization of the already-selected clusters
- **Heatmap of active predictor-by-cluster coefficients**, to show which predictors matter more in different firm types
- **Ranked table of the most important interaction terms**, to connect the clustered model back to interpretable finance relations

### Important note on PCA
PCA is used here only for visualization. It helps display the selected clusters in two dimensions, but it is **not** used to define the clusters themselves.

# 5. Portfolio Construction

This section translates the monthly stock-level return predictions into investable portfolio positions. Prediction accuracy and portfolio performance are related but distinct: a model can improve cross-sectional forecasting metrics without necessarily producing a strong trading strategy, and vice versa. We therefore treat portfolio construction as a separate step after the final prediction models have been estimated and evaluated.

## Portfolio Rule

### Objective
Define how monthly predicted returns are mapped into portfolio weights.

### Main rule
For each test-period month:
- rank stocks by predicted return,
- long the top decile,
- short the bottom decile,
- equal-weight within each leg,
- impose dollar neutrality.

### Why we use this rule
- it is simple and transparent,
- it is comparable across models,
- it avoids unstable raw-weight scaling from noisy monthly forecasts,
- it is easy to explain in the report and presentation.

### What this section should do
- take monthly stock-level predictions from each model,
- sort stocks into predicted-return groups,
- assign long and short positions,
- compute monthly portfolio weights.

### Useful visualizations
- schematic of the prediction-to-portfolio mapping,
- monthly number of stocks in long and short legs,
- distribution of predicted scores before portfolio sorting.

## Portfolio Performance

### Objective
Evaluate the economic performance of the portfolios generated by each prediction model.

### What this section should do
- compute monthly long-short portfolio returns for each model,
- summarize performance over the test period,
- compare the investment implications of the benchmark and ML models.

### Required output
Report:
- annualized mean excess return,
- annualized volatility,
- Sharpe ratio.

### Models compared
- Historical-average benchmark portfolio
- OLS-based portfolio
- Pooled Elastic Net portfolio
- Clustered Elastic Net portfolio

### Useful visualizations
- cumulative return curves across models,
- bar chart of annualized return / volatility / Sharpe,
- rolling Sharpe or rolling cumulative return,
- drawdown plot.

## Practical Trading Discussion

### Objective
Discuss whether the portfolio strategy is economically plausible after implementation frictions.

### What this section should do
Provide a qualitative discussion of:
- turnover,
- transaction costs,
- liquidity constraints,
- concentration risk,
- whether the strategy is likely to survive realistic trading frictions.

### Why we do it
The project brief explicitly asks for a qualitative discussion of implementation issues, even if we do not model them quantitatively.

### Useful visualizations
- turnover plot over time, if turnover is computed,
- average holding-count summary,
- concentration or weight-distribution plot, if useful.